In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F



import torch.nn.functional as F

from lib.octree_modified import Octree

from models.networks.dualoctree_networks.graph_sem_vae import GraphVAE


model = GraphVAE(
    depth=6,
    channel_in=32,     # same as embedding_dim
    nout=2,
    full_depth=3,
    depth_stop=4,
    depth_out=6,
    embed_dim=8,       # latent dimension
    num_classes=27,
    resblk_num = 3,
).cuda()

In [2]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import models.networks.dualoctree_networks.dual_octree as dual_octree
from lib.octree import Octree
from lib.dual_octree import DualOctree
from lib.octreed import OctreeD
import tqdm
from utils.util_octree_stuff import *
import os
import torch
from torch.utils.data import Dataset, DataLoader

class VideoVoxelDataset(Dataset):
    def __init__(self, base_path):
        self.paths = sorted([os.path.join(base_path, f) 
                             for f in os.listdir(base_path)
                             if f.endswith(".pt")])
        assert len(self.paths) > 0, f"No .pt files found in {base_path}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        voxel = torch.load(path)                         # (T,H,W) uint8
        voxel = voxel.long()                             # convert to long labels
        mask = (voxel != 0)                              # boolean mask
        return voxel, mask, os.path.basename(path)       # return name for logging


def collate_fn(batch):
    # batch is list of tuples, but we process *one at a time*
    return batch[0]  # return (voxel, mask, name)
dataset = VideoVoxelDataset("/home/zxj/Documents/dataset/softmotion30_44k/26_vote")
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)


depth = 6
step_counter=0

from models.networks.dualoctree_networks.dual_octree import DualOctree
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# depth = 4
model.train()
for epoch in range(5000):
    pbar = tqdm.tqdm(dataloader, desc=f"Epoch {epoch}")
    total_loss_sum = 0.0
    sem_loss_sum = 0.0
    acc_sum = 0.0
    valid_rate_sum = 0.0 
    count = 0
    for voxel_tensor, voxel_mask, fname in pbar:
        
        voxel_tensor = voxel_tensor.cuda()
        voxel_mask = voxel_mask.cuda()
        


   

        octree = Octree(depth=6, full_depth=3)
        padded_sem = 26
        voxel_tensor_zero = torch.ones((64,64,64)) * padded_sem
        voxel_tensor_zero[0:30,:,:] = voxel_tensor

       
        voxel_tensor = voxel_tensor_zero.cuda()
        points = voxel_grid_to_points(voxel_tensor,use_semantic_octree=True)
        octree.build_octree_with_semantics(points,voxel_tensor)

        octree = octree.cuda()
        weight_sem = 1
        sem_loss = False
        model.train()
        output = model(octree, octree_out=octree, pos=None, evaluate=False, update_octree=False)
        
        z ,_= model.extract_code(octree)
        # doctree = DualOctree(octree)
        # doctree.post_processing_for_docnn()
        # output = model.decode_code(torch.randn_like(z),doctree,False)


        logits = output['logits']
        sem_voxs = output['sem_voxs']
        sem_vox = output['sem_vox']

        
        
        loss_dict = compute_octree_loss(logits, octree) 
        # sem_1 = octree.semantic_flatten[1][octree.semantic_flatten[1]  !=-1]
        # sem_2 = octree.semantic_flatten[2][octree.semantic_flatten[2]  !=-1]
        sem_2 = torch.tensor([]).cuda()
        sem_3 = octree.semantic_flatten[3][octree.semantic_flatten[3]  !=-1]
        sem_4 = octree.semantic_flatten[4][octree.semantic_flatten[4] !=-1]
        sem_5 = octree.semantic_flatten[5][octree.semantic_flatten[5] !=-1]
        sem_6 = octree.semantic_flatten[6]
        # sems_total = torch.cat([sem_2,sem_3,sem_4,sem_5,sem_6])
        # sem_loss = (F.cross_entropy(sem_vox,sems_total.long(), reduction='mean'))
        # 拼接标签
        sems_total = torch.cat([sem_3, sem_4, sem_5, sem_6])
        # 对应每层的权重
        # alpha = 0.125
        alpha = 0.125
        weights_list = [math.exp(-alpha*(d-3)) for d in [3,4,5,6]] *  8*8
        weights = torch.cat([
            torch.full_like(sem_3.float(), weights_list[0]),
            torch.full_like(sem_4.float(), weights_list[1]),
            torch.full_like(sem_5.float(), weights_list[2]),
            torch.full_like(sem_6.float(), weights_list[3]),
        ])
        weights = weights / weights.mean()  # 可选：归一化避免loss过小

        # weighted CE
        kl_loss = output['kl_loss']
        octree_out = output['octree_out']
        sem_loss = F.cross_entropy(sem_vox, sems_total.long(), reduction='none')
        sem_loss = (sem_loss * weights).mean()

        total_loss = 0.1 * kl_loss
        for k in loss_dict:
            if 'loss_' in k:
                total_loss += loss_dict[k]
        total_loss += sem_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        count += 1
        with torch.no_grad():
            alpha = 0.125
            levels = [3, 4, 5, 6]
            weights_list = [math.exp(-alpha*(d-3)) for d in levels]
            weights_tensor = torch.tensor(weights_list, device='cuda')
            weights_tensor = weights_tensor / weights_tensor.sum()

            offset = 0
            accs = []
            total_correct = 0.0
            total_weight = 0.0

            for i, w in zip(levels, weights_tensor):
                gt = octree.semantic_flatten[i][octree.semantic_flatten[i] != -1]
                length = gt.shape[0]
                if length == 0:
                    accs.append(0.0)
                    continue

                preds = sem_vox[offset:offset + length]
                offset += length

                pred_cls = preds.argmax(dim=1)
                correct = (pred_cls == gt).sum().item()
                acc = correct / float(length)

                accs.append(acc)
                total_correct += w * correct
                total_weight += w * length

            acc_weighted = total_correct / total_weight if total_weight > 0 else 0.0

            # ✅ 打印在 tqdm 上
            pbar.set_postfix({
                "total_avg": f"{total_loss_sum / count:.4f}",
                "kl loss": f"{kl_loss:.4f}",
                "sem loss": f"{sem_loss:.4f}",
                "acc3": f"{accs[0]:.3f}",
                "acc4": f"{accs[1]:.3f}",
                "acc5": f"{accs[2]:.3f}",
                "acc6": f"{accs[3]:.3f}",
                "acc_w": f"{acc_weighted:.3f}",
            })
        break
    break




Epoch 0:   0%|          | 0/169 [00:00<?, ?it/s]/tmp/ipykernel_11660/3100973806.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  voxel = torch.load(path)                

TypeError: GraphVAE.forward() got an unexpected keyword argument 'evaluate'

In [3]:
import torch
import numpy as np
import imageio

def save_gif_from_indices(idx, codebook, out_path, fps=12):
    """
    Render a voxel/video index tensor to an RGB GIF using a codebook.
    Invalid indices are shown as solid black.

    Args:
        idx: torch.Tensor or np.ndarray, shape [T, H, W]
             Each value is a semantic index (int or float convertible to int).
        codebook: [K, 3] tensor or np.ndarray, RGB in [0,1]
        out_path: path to output .gif
        fps: frame rate
    """
    # --- 1️⃣ Convert to torch
    if isinstance(idx, np.ndarray):
        idx = torch.from_numpy(idx)
    if isinstance(codebook, np.ndarray):
        codebook = torch.from_numpy(codebook)

    idx = idx.long()
    codebook = codebook.float()

    # --- 2️⃣ Build RGB(A)
    T, H, W = idx.shape
    K = codebook.shape[0]
    valid_mask = (idx >= 0) & (idx < K)

    rgb = torch.zeros((T, H, W, 4), dtype=torch.float32, device=idx.device)
    rgb[..., :3][valid_mask] = codebook[idx[valid_mask]]
    rgb[..., 3] = 1.0  # fully opaque everywhere (no transparency)

    # --- 3️⃣ Convert to uint8 RGB
    rgb = (rgb * 255).clamp(0, 255).byte().cpu().numpy()

    # --- 4️⃣ Save as GIF
    imageio.mimsave(out_path, rgb, fps=fps)
    print(f"✅ Saved GIF → {out_path}, shape={rgb.shape}, fps={fps}")


In [4]:
codebook = np.load("bair_palette_K.npy")
codebook.shape

(26, 3)

In [5]:
save_gif_from_indices(octree.get_semantic_voxels(padded_sem).cpu().numpy(), codebook, "ori.gif", fps=8)

✅ Saved GIF → ori.gif, shape=(64, 64, 64, 4), fps=8


In [6]:
offset = 0
sem_list = [sem_3,sem_4,sem_5,sem_6]
for i in range(3,6):
    # print(i)
    length = octree.semantic_flatten[i][octree.semantic_flatten[i]  !=-1].shape[0]
    # octree.semantic_flatten[i] = torch.argmax(sem_vox[offset:offset+length], dim = 1)
    # octree.semantic_flatten[i] = sem_list[i-3]
    sem_recon = -torch.ones_like(octree.semantic_flatten[i])
    sem_mask = octree.children[i] == -1
    sem_recon[sem_mask] = torch.argmax(sem_vox[offset:offset+length], dim = 1).float()
    octree.semantic_flatten[i] = sem_recon
    offset+=length
    # print(length)
octree.semantic_flatten[6] = torch.argmax(sem_vox[offset:], dim = 1).float()

In [7]:
voxel_recon = octree.get_semantic_voxels(padded_sem)


In [8]:
save_gif_from_indices(voxel_recon.cpu().numpy(), codebook, "recon.gif", fps=8)

✅ Saved GIF → recon.gif, shape=(64, 64, 64, 4), fps=8
